# Finanzas I · Unidad II | Tema 79
**Universidad Nacional del Centro del Perú · Facultad de Economía**  
**Autora:** Amy Pamela Guadalupe Cordova · 2024200501G  
**Docente:** Dr. Ciro Iván Machacuay Meza · 2026-II

## Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
Objetivo: comparar el desempeño ajustado por riesgo de los fondos mutuos peruanos. Se conserva la estructura sencilla de la versión 0.3 y de la Unidad I; no se importa ningún archivo ni dato de esos trabajos. La SMV rechazó robots.txt con HTTP 418 el 19/11/2026 a las 11:47; este cuaderno no contacta esa fuente. La segunda vía elegida es el catálogo público de boletines mensuales de la AAF, que contiene rentabilidades y valores cuota de fondos individuales. Se realiza una extracción nueva. La documentación de la incidencia y, cuando corresponda, la confirmación escrita de la alternativa deben completarse con evidencia auténtica. No se declara autorización obtenida.

**Importante:** el catálogo y tablas de boletines 2022 y 2025 existen públicamente; el boletín de enero de 2025 se probó localmente con 259 filas de fondos/series y 0 descartes entre las filas candidatas detectadas (258 valores cuota legibles; una cuota presenta separadores ambiguos); los demás meses y la ejecución completa en Colab quedan pendientes. El piloto del paso 4 debe confirmar legibilidad antes de descargar el resto. No continuar si falla. El periodo objetivo es enero de 2019 a diciembre de 2025, mensual; no se inventan datos diarios. El mínimo operativo es más de 1 000 observaciones válidas por variable y más de 1 000 filas completas del modelo; la cobertura efectiva se reporta, y no se afirma cumplir automáticamente la cobertura diaria sugerida en el temario. Los boletines no aportan necesariamente patrimonio ni partícipes por fondo, por lo que no se presentan como observaciones individuales. Se prueba la reproducción local y se distingue de la fuente viva.

### Diseño financiero
Pregunta: ¿cómo difieren rentabilidad, volatilidad y Sharpe entre tipos de fondos y qué factores se asocian con la rentabilidad mensual? Hipótesis: las categorías de mayor riesgo presentan mayor dispersión; un mayor riesgo rezagado no garantiza un mejor Sharpe; la rentabilidad rezagada puede asociarse con persistencia o reversión del retorno, y el factor cambiario puede tener signos distintos según la cartera.

| Variable | Rol | Fuente y construcción |
|---|---|---|
| Rentabilidad mensual (`rent_mes`) | Y | AAF, retorno publicado en moneda del fondo, decimal |
| `vol12` | X1 | Desviación estándar de retornos mensuales anteriores, ventana calendario de 12 meses, mínimo 6 |
| `var_tc` | X2 | BCRP `PN01216PM`, diferencia logarítmica del TC interbancario promedio de fin de mes |
| `rent_lag1` | X3 | Rentabilidad publicada del mes calendario anterior, decimal; evalúa persistencia |
| `rf_mensual` | Indicador auxiliar | BCRP `PD04692MD` / `PD04693MD`, tasa interbancaria por moneda, aproximación mensual |

Unidad: denominación publicada de fondo/serie y moneda por mes; pueden existir clases de un mismo fondo, no se confunden con fondos únicos. Unión por mes. Modelo MCO con tres factores y controles categóricos de tipo/moneda; inferencia agrupada por unidad y mes. La ventana de volatilidad y el rezago reducen el número de meses útiles. El boletín no contiene comisiones; no se imputan ni se incorporan al modelo. Las observaciones mensuales del TC se repiten legítimamente en el panel: se informa también el número de meses independientes. Se conservan caídas reales, se marcan atípicos y se evalúa sensibilidad. Sharpe por unidad y moneda, no compara directamente el sol con el dólar; se usa tasa proxy, no activo literalmente libre de riesgo. La anualización supone estabilidad de retornos y no representa el rendimiento acumulado de meses no observados. Retornos publicados no incorporan necesariamente comisiones de suscripción/rescate, impuestos ni distribuciones; no se afirma retorno total. Asociación, no causalidad. Referencias teóricas: Markowitz (1952), *Portfolio Selection*, DOI 10.1111/j.1540-6261.1952.tb01525.x; Sharpe (1994), *The Sharpe Ratio*, DOI 10.3905/jpm.1994.409501; Cameron, Gelbach y Miller (2011), *Robust Inference With Multiway Clustering*, DOI 10.1198/jbes.2010.07136. Son fundamentos, no sustituyen el estado del arte reciente.

**Cómo usar:** abre este cuaderno en Colab y ejecuta los pasos en orden. Descarga el ZIP antes de cerrar sesión (el almacenamiento es temporal). GitHub se publica voluntariamente al final. Para el profesor basta clonar el repositorio, instalar requirements y ejecutar 05 en modo local. No conecta a Drive.

## Paso 1. Crear carpeta de trabajo e instalar librerías

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata as meta
from datetime import datetime, timezone, timedelta
RAIZ = Path('/content/3_Base_de_datos_y_codigo_Amy_Pamela_Guadalupe_Cordova')
RAIZ.mkdir(parents=True, exist_ok=True)
os.chdir(RAIZ)
for carpeta in ['codigo','datos_crudos','datos_procesados','salidas','documentos']:
    (RAIZ/carpeta).mkdir(exist_ok=True)
# Versiones comprobadas en el entorno local de desarrollo; Colab verifica al instalar.
VERSIONES = {'requests':'2.32.5','pandas':'2.2.3','numpy':'2.3.5','scipy':'1.17.0','statsmodels':'0.14.6','matplotlib':'3.10.8','beautifulsoup4':'4.14.3','pdfplumber':'0.11.9'}
Path('requirements.txt').write_text(''.join(f'{k}=={v}\n' for k,v in VERSIONES.items()),encoding='utf-8')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-c','import requests,pandas,numpy,scipy,statsmodels.formula.api,matplotlib.pyplot,bs4,pdfplumber; print("Importaciones verificadas")'],check=True)
for k,v in VERSIONES.items():
    if meta.version(k)!=v: raise RuntimeError(f'{k}: versión instalada distinta')
# Los cinco scripts se ejecutan en procesos nuevos; no se rechazan módulos ya cargados en el kernel.
print('Python:',sys.version.split()[0], '| carpeta:',RAIZ)

## Paso 2. Configuración y control de ejecución

In [ ]:
import json
ESTADO = {}
HORA_PERU = timezone(timedelta(hours=-5))
def correr(script, requiere=(), argumentos=()):
    for previo in requiere:
        if ESTADO.get(previo)!=0: raise RuntimeError('Primero debe terminar correctamente '+previo)
    p = subprocess.Popen([sys.executable,'-u',str(RAIZ/'codigo'/script),*argumentos],cwd=RAIZ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout: print(linea,end='',flush=True)
    p.wait()
    ESTADO[script]=p.returncode
    (RAIZ/'documentos/estado_ejecucion.json').write_text(json.dumps(ESTADO,indent=2),encoding='utf-8')
    if p.returncode: raise RuntimeError(f'{script} terminó con código {p.returncode}; revisa el log y no continúes.')
Path('.env.example').write_text('# Token solo para publicar desde Colab. No se necesita para reproducir.\nGITHUB_TOKEN=\n',encoding='utf-8')
Path('.gitignore').write_text('.env\n__pycache__/\n.ipynb_checkpoints/\n*.pyc\n',encoding='utf-8')
Path('enlace_repositorio_github.txt').write_text('https://github.com/AmyGuadalupe/Finanzas-Segundo-Trabajo\n',encoding='utf-8')
Path('incidencias_fuente.md').write_text('# Incidencias de fuente\n- SMV: `https://www.smv.gob.pe/robots.txt`, 19/11/2026, 11:47 (Lima), HTTP 418, según captura de Colab de la estudiante. El programa anterior se detuvo. Este proyecto no vuelve a consultar la SMV ni cambia de conexión para eludir restricciones.\n- Vía alternativa propuesta: boletines mensuales públicos de AAF, https://fondosmutuos.pe/estadisticas/. Se preserva la evidencia original descargada y las reglas de rastreo. No consta en este archivo autorización concedida; adjuntar respuesta real si corresponde al numeral 2.4.3.\n- Añadir la captura original a documentos/ y registrar la comunicación real. Los nuevos errores y el resultado del parseo se registran en el log y en los archivos de control; no se rellenan meses faltantes.\n',encoding='utf-8')
Path('log_ejecucion.txt').touch(exist_ok=True)
# Si se ejecutó el piloto descartado de informes en esta misma sesión, se conserva aparte.
import shutil
for anterior in [Path('datos_crudos/scraping/pdf'), Path('datos_crudos/scraping/descargas_2024200501G.csv')]:
    if anterior.exists():
        respaldo=Path('documentos/piloto_informe_no_utilizado')/anterior.name
        respaldo.parent.mkdir(parents=True,exist_ok=True)
        if not respaldo.exists(): shutil.move(str(anterior),str(respaldo))


## Paso 3. API BCRP: tipo de cambio y tasas por moneda

In [ ]:
%%writefile codigo/01_extraccion_api.py
# Autora: Amy Pamela Guadalupe Cordova
# Código de matrícula: 2024200501G
# Tema 79: Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
# Fecha de extracción: 2026-11-19
"""Vía API BCRPData. Una consulta por serie; guarda JSON original y CSV.
--local: reconstruye exclusivamente desde los JSON incluidos. --demo: consulta
el tramo de cierre y coteja diez valores disponibles, sin sobrescribir crudos.
"""
import argparse, hashlib, json, time, re
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import requests

RAIZ = Path(__file__).resolve().parents[1]
DIR = RAIZ / 'datos_crudos' / 'api'
DIR.mkdir(parents=True, exist_ok=True)
FECHA_INICIO = '2018-12-01'  # mes previo para variación de enero de 2019
FECHA_CORTE = '2025-12-31'
BASE = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'
UA = 'UNCP-FinanzasI/1.0 (investigacion academica; contacto: e_2024200501g@uncp.edu.pe)'
SERIES = {
    'PN01216PM': ('tc_fin', 'mensual', 'interbancario', 'S/ por USD'),
    'PD04692MD': ('tasa_pen', 'diaria', 'interbancaria', '% anual'),
    'PD04693MD': ('tasa_usd', 'diaria', 'interbancaria', '% anual'),
}
MESES = {m:i for i,m in enumerate('ene feb mar abr may jun jul ago set oct nov dic'.split(),1)}
MESES['sep'] = 9

def anotar(s):
    linea = f'{datetime.now(timezone(timedelta(hours=-5))):%Y-%m-%d %H:%M:%S} | 01_api | {s}'
    print(linea)
    with (RAIZ/'log_ejecucion.txt').open('a',encoding='utf-8') as f: f.write(linea+'\n')

def url(codigo, inicio=FECHA_INICIO, fin=FECHA_CORTE):
    if SERIES[codigo][1]=='mensual': inicio,fin=inicio[:7],fin[:7]
    return f'{BASE}/{codigo}/json/{inicio}/{fin}/esp'

def consultar(direccion):
    time.sleep(1.5)
    r=requests.get(direccion,headers={'User-Agent':UA},timeout=90)
    anotar(f'HTTP {r.status_code} | {direccion}')
    r.raise_for_status()
    return r.content

def fecha(texto):
    # Formatos BCRP: Ene.2019, Ene19 y 02.Ene.2019.
    s=str(texto).strip().lower()
    m=re.fullmatch(r'(?:(\d{1,2})[. /-]*)?([a-záéíóú]{3})[. /-]*(\d{2,4})',s)
    if m:
        d,mes,a=m.groups(); a=int(a); a=a+2000 if a<100 else a
        return pd.Timestamp(a,MESES[mes],int(d or 1))
    return pd.to_datetime(s,format='%Y-%m-%d',errors='raise')

def leer(codigo,contenido):
    j=json.loads(contenido.decode('utf-8-sig'))
    nombre=j['config']['series'][0]['name']
    clave,frec,esperado,unidad=SERIES[codigo]
    if esperado not in nombre.lower(): raise ValueError(f'{codigo}: nombre inesperado {nombre}')
    if codigo=='PD04692MD' and 'S/' not in nombre: raise ValueError(nombre)
    if codigo=='PD04693MD' and 'US$' not in nombre: raise ValueError(nombre)
    filas=[{'codigo':codigo,'variable':clave,'fecha':fecha(p['name']), 'periodo_original':p['name'],
            'valor_original':p['values'][0], 'nombre_serie':nombre,'unidad':unidad,'frecuencia':frec}
           for p in j['periods']]
    d=pd.DataFrame(filas).sort_values('fecha')
    if d.empty or d.fecha.duplicated().any(): raise ValueError(f'{codigo}: sin datos o fechas duplicadas')
    return d

def main(local=False,demo=False):
    bloques=[]
    for codigo in SERIES:
        archivo=DIR/f'{codigo}.json'
        if demo:
            nuevo=leer(codigo,consultar(url(codigo,'2025-12-01','2025-12-31')))
            viejo=leer(codigo,archivo.read_bytes())
            cotejo=nuevo.merge(viejo,on=['codigo','fecha'],suffixes=('_vivo','_crudo'),validate='one_to_one')
            if cotejo.empty: raise ValueError('Sin fechas comunes para cotejo')
            igual=cotejo.valor_original_vivo.astype(str)==cotejo.valor_original_crudo.astype(str)
            print(cotejo[['fecha','valor_original_vivo','valor_original_crudo']].head(10).to_string(index=False))
            anotar(f'DEMO {codigo}: {igual.sum()}/{len(igual)} coincidencias exactas; revisiones se documentan')
            continue
        contenido=archivo.read_bytes() if local else consultar(url(codigo))
        d=leer(codigo,contenido)
        if not d.fecha.between(FECHA_INICIO,FECHA_CORTE).all(): raise ValueError(f'{codigo}: periodo fuera de corte')
        if d.fecha.dt.to_period('M').nunique()<85: raise ValueError(f'{codigo}: cobertura temporal insuficiente')
        if not local:
            archivo.write_bytes(contenido)
            anotar(f'GUARDADO {archivo.relative_to(RAIZ)} | {len(d)} filas | SHA256 {hashlib.sha256(contenido).hexdigest()}')
        bloques.append(d)
    if not demo:
        pd.concat(bloques,ignore_index=True).to_csv(RAIZ/'datos_crudos/datos_crudos_2024200501G_bcrp.csv',index=False,date_format='%Y-%m-%d')
        anotar(f'CSV BCRP: {sum(map(len,bloques))} filas originales')

if __name__=='__main__':
    p=argparse.ArgumentParser(); g=p.add_mutually_exclusive_group(); g.add_argument('--local',action='store_true');g.add_argument('--demo',action='store_true'); a=p.parse_args(); main(a.local,a.demo)


In [ ]:
# Si se conservan los JSON auténticos de la extracción de esta misma sesión, no volver a pedirlos.
import hashlib
log_actual = Path('log_ejecucion.txt').read_text(encoding='utf-8')
api_actual = [Path('datos_crudos/api') / (c+'.json') for c in ['PN01216PM','PD04692MD','PD04693MD']]
if all(p.is_file() and ('SHA256 '+hashlib.sha256(p.read_bytes()).hexdigest()) in log_actual for p in api_actual):
    correr('01_extraccion_api.py', argumentos=('--local',))
else:
    correr('01_extraccion_api.py')

## Paso 4. Descarga programática: boletines AAF
Primero se descarga y revisa un boletín piloto de enero de 2025. Si no reconoce las tablas, detiene la etapa. Revisa el control de parseo.

In [ ]:
%%writefile codigo/02_scraping_web.py
# Autora: Amy Pamela Guadalupe Cordova
# Código de matrícula: 2024200501G
# Tema 79: Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
# Fecha de extracción: 2026-11-19
"""Vía 2: boletines mensuales públicos AAF. Una página de catálogo, un PDF por mes.
No se consulta la SMV (HTTP 418 documentado). --local reparsea PDF entregados.
--demo coteja un PDF vivo con su copia original. No usa Drive ni sesiones previas.
"""
import argparse, hashlib, re, time, unicodedata
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests
import pdfplumber
from bs4 import BeautifulSoup

RAIZ=Path(__file__).resolve().parents[1]
DIR=RAIZ/'datos_crudos'/'scraping'; PDF=DIR/'pdf_boletines'; PDF.mkdir(parents=True,exist_ok=True)
FECHA_INICIO='2019-01'; FECHA_CORTE='2025-12'
LISTADO='https://fondosmutuos.pe/estadisticas/'
ROBOTS='https://fondosmutuos.pe/robots.txt'
PILOTO='https://fondosmutuos.pe/wp-content/uploads/2025/02/Boletin-Enero-2025-9.pdf'
UA='UNCP-FinanzasI/1.0 (investigacion academica; contacto: e_2024200501g@uncp.edu.pe)'
MANIFIESTO=DIR/'descargas_boletines_2024200501G.csv'
MESES={m:i for i,m in enumerate('enero febrero marzo abril mayo junio julio agosto septiembre octubre noviembre diciembre'.split(),1)}; MESES['setiembre']=9
TIPOS=[('Fondo de fondos',r'fondos?\s+de\s+fondos'),('Estructurados',r'estructurad'),('Acciones',r'acciones|renta variable'),('Mixtos',r'mixt'),('Flexibles',r'flexib'),('Deuda',r'deuda|renta fija')]
# Identificación auxiliar; si no hay coincidencia se conserva la denominación completa de la fuente.
ADMIN=['BBVA Asset Management','Credicorp Capital','Interfondos','Scotia Fondos','Fondos Sura','Sura','Diviso Fondos','Diviso','Independiente','Promoinvest','Faro Capital','Flip SAF','Flip','Blum','Andino','Prival','Compass','Pacífico','Pacifico','Grupo Coril','Coril','Popular SAFI','Popular','Continental','Seminario']

def anotar(s):
    linea=f'{datetime.now(timezone(timedelta(hours=-5))):%Y-%m-%d %H:%M:%S} | 02_aaf | {s}'
    print(linea)
    with (RAIZ/'log_ejecucion.txt').open('a',encoding='utf-8') as f:f.write(linea+'\n')

def plano(s):
    return ''.join(c for c in unicodedata.normalize('NFD',str(s)) if unicodedata.category(c)!='Mn').lower()

def periodo(s):
    s=plano(s)
    for m,n in MESES.items():
        x=re.search(rf'\b{m}[-_ /]*?(20\d\d)\b',s)
        if x:return f'{x[1]}-{n:02d}'
    return None

def numero(s,porcentaje=False):
    s=s.strip().replace('%','').replace('−','-').replace(' ','')
    if s.lower() in ('-','–','—','n/a','nd','n.d.',''):return None
    if ',' in s and '.' in s:
        s=s.replace('.','').replace(',','.') if s.rfind(',')>s.rfind('.') else s.replace(',','')
    else:s=s.replace(',','.')
    try:return float(s)/ (100 if porcentaje else 1)
    except ValueError:return None

# Mes del reporte: se exige una fecha explícita de portada; no se infiere de las filas.
ABREV={'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,'jul':7,'ago':8,'sep':9,'set':9,'oct':10,'nov':11,'dic':12}
def mes_portada(texto):
    t=plano(texto)
    for linea in t.splitlines():
        for m,n in {**MESES,**ABREV}.items():
            x=re.fullmatch(rf'(?:reporte mensual\s+)?{m}\.?[\s/-]+(20\d\d)(?:\s+.*)?',linea.strip())
            if x:return f'{x[1]}-{n:02d}'
    return None

# Las líneas de títulos determinan las categorías, incluso en continuaciones de página.
CABECERAS=[('Fondo de fondos',r'(?:fondos? de fondos)(?: en (?:soles|dolares))?'),('ETF',r'(?:fondo bursatil(?: etf)?|etf)'),('Estructurados',r'(?:fondos? )?estructurados?(?: en (?:soles|dolares))?'),('Acciones',r'(?:de )?acciones(?: en (?:soles|dolares))?'),('Mixtos',r'mixtos?(?: en (?:soles|dolares))?'),('Flexibles',r'flexibles?(?: en (?:soles|dolares))?'),('Deuda',r'(?:de )?deuda(?: en (?:soles|dolares)|\s*\([^)]*\))?')]
ADMIN += ['Andean Crown','Prudential SAF','El Dorado','BTG Pactual Perú','Blum SAF','Coril SAF','BD Capital','Faro Capital SAFI','AC SAF','MCC','Sura SAF','Adcap','Credicorp Capital SAF']
N=r'[-+]?\d[\d.,]*'; FALTA=r'(?:n/a|n\.d\.|nd|[-–—])'
PATRON=re.compile(rf'^(?P<nombre>.+?)\s+(?P<mon>US\s*\$|S/\.?|USD|PEN|\$)\s+(?P<vc>{N}|{FALTA})\s+(?P<ret>{N}\s*%|{FALTA})(?:\s|$)',re.I)
def leer_pdf(ruta,mes):
    filas=[]; descartes=[]; candidatos=0; tipo=None
    with pdfplumber.open(ruta) as f:
        portada=f.pages[0].crop((0,0,f.pages[0].width,f.pages[0].height)).extract_text() or ''
        declarado=mes_portada(portada)
        if declarado!=mes:raise ValueError(f'Portada {declarado!r} no coincide con {mes}')
        for i,p in enumerate(f.pages,1):
            # Excluir texto situado fuera del área visible: el boletín de enero 2025 contiene texto desbordado de otras páginas.
            texto=p.crop((0,0,p.width,p.height)).extract_text(x_tolerance=2,y_tolerance=2) or ''
            if i <= 2 and 'valor cuota' not in plano(texto):continue
            for linea in texto.splitlines():
                s=unicodedata.normalize('NFKC',linea).replace('−','-')
                s=' '.join(''.join(' ' if unicodedata.category(c) in ('Co','So') else c for c in s).split())
                titulo=plano(s)
                # Algunos encabezados incluyen la segunda línea de las columnas.
                titulo=re.sub(r'\s+mes\s+mes\s+20\d\d\s+ultimos\s+12\s+meses$','',titulo)
                nuevo=next((k for k,pat in CABECERAS if re.fullmatch(pat,titulo)),None)
                if nuevo:tipo=nuevo;continue
                if not tipo:continue
                if titulo.startswith(('nota:','nota :','disclaimer')):tipo=None;continue
                m=PATRON.match(s)
                if not m:
                    if re.search(r'\s(?:US\s*\$|S/\.?|USD|PEN)\s+\d',s,re.I):
                        candidatos+=1;descartes.append({'mes':mes,'pagina':i,'archivo':ruta.name,'fila_original':linea})
                    continue
                candidatos+=1
                nombre=m['nombre'].strip(); vc=numero(m['vc']); rm=numero(m['ret'],True)
                # No adivinar números de cuota con separadores defectuosos. Conservar el retorno publicado.
                if m['vc'].count('.')>1 or m['vc'].count(',')>1 and '.' not in m['vc']:vc=None
                if rm is None or rm < -1:
                    descartes.append({'mes':mes,'pagina':i,'archivo':ruta.name,'fila_original':linea});continue
                safm='';fondo=nombre
                for a in sorted(ADMIN,key=len,reverse=True):
                    if plano(fondo).endswith(' '+plano(a)):
                        safm=a;fondo=fondo[:-len(a)].strip();break
                filas.append({'mes':mes,'tipo':tipo,'fondo':fondo,'administradora':safm,'denominacion_original':nombre,'moneda':'USD' if '$' in m['mon'] or m['mon'].upper()=='USD' else 'PEN','estado':'no informado','valor_cuota':vc,'rent_mes':rm,'archivo':ruta.name,'pagina':i,'fila_original':linea})
    return filas,descartes,candidatos

def pedir(url,reglas=None,es_robots=False):
    # Seguir únicamente redirecciones normales dentro del mismo portal; nunca un rechazo.
    for _ in range(4):
        if urlparse(url).hostname not in ('fondosmutuos.pe','www.fondosmutuos.pe'):raise ValueError('Dominio no autorizado: '+url)
        if reglas is not None and not reglas.can_fetch(UA,url):raise RuntimeError('robots.txt restringe '+url)
        time.sleep(1.5)
        r=requests.get(url,headers={'User-Agent':UA},timeout=90,allow_redirects=False)
        anotar(f'HTTP {r.status_code} | {url}')
        if r.status_code in (401,403,418,429):
            (DIR/'bloqueo_aaf.txt').write_text(f'HTTP {r.status_code} | {url}',encoding='utf-8')
            raise RuntimeError('Acceso rechazado: no se reintenta ni se cambia de conexión.')
        if r.status_code in (301,302,303,307,308) and r.headers.get('Location'):
            url=urljoin(url,r.headers['Location']);continue
        if es_robots and r.status_code in (404,410):
            anotar('robots.txt no está publicado (HTTP 404/410); se conserva registro y se respetan los rechazos de cada URL.')
            return b'# REGISTRO LOCAL: robots.txt no publicado; ver HTTP en log.\n'
        if r.status_code!=200:raise RuntimeError(f'HTTP {r.status_code}; {url}')
        return r.content
    raise RuntimeError('Demasiadas redirecciones del portal.')

def reglas_portal():
    if (DIR/'bloqueo_aaf.txt').exists():raise RuntimeError('Existe un rechazo de AAF registrado; no continuar sin revisar autorización.')
    contenido=pedir(ROBOTS,es_robots=True); (DIR/'robots_aaf.txt').write_bytes(contenido)
    r=RobotFileParser();r.parse(contenido.decode('utf-8',errors='replace').splitlines());return r

def enlaces(contenido):
    sopa=BeautifulSoup(contenido,'html.parser'); candidatos={}
    for a in sopa.find_all('a',href=True):
        href=urljoin(LISTADO,a['href']); etiqueta=plano(a.get_text(' ',strip=True))
        if not urlparse(href).path.lower().endswith('.pdf'):continue
        if urlparse(href).hostname not in ('fondosmutuos.pe','www.fondosmutuos.pe'):continue
        if 'boletin' not in etiqueta and 'boletin' not in plano(urlparse(href).path):continue
        h=a.find_previous(['h1','h2','h3','h4','h5','h6'])
        mes=periodo(h.get_text(' ',strip=True)) if h else None
        mes_url=periodo(urlparse(href).path.rsplit('/',1)[-1])
        if mes and mes_url and mes!=mes_url:continue
        mes=mes or mes_url
        if mes and FECHA_INICIO<=mes<=FECHA_CORTE:
            candidatos.setdefault(href,set()).add(mes)
    links={}
    for href,meses in candidatos.items():
        # Las versiones móvil/escritorio pueden repetir enlaces de otro mes.
        if len(meses)!=1:continue
        mes=next(iter(meses));links.setdefault(mes,href)
    links['2025-01']=PILOTO
    return links

def procesar(manifiesto):
    todas=[]; descartes=[]; control=[]
    for row in manifiesto.sort_values('mes').itertuples():
        ruta=PDF/row.archivo
        if hashlib.sha256(ruta.read_bytes()).hexdigest()!=row.sha256:raise ValueError('PDF alterado: '+row.archivo)
        try:
            f,d,c=leer_pdf(ruta,row.mes)
        except Exception as e:
            anotar(f'{row.mes} | parseo fallido: {e}');f,d,c=[],[],0
        aceptable=len(f)>=25 and c>0 and len(f)/c>=.90
        if aceptable:todas+=f
        else:anotar(f'{row.mes}: boletín excluido por calidad del parseo; revisar original.')
        descartes+=d
        control.append({'mes':row.mes,'archivo':row.archivo,'url':row.url,'filas':len(f),'candidatos':c,'no_reconocidas':len(d),'aceptado':aceptable})
        anotar(f'{row.mes} | {len(f)} filas reconocidas de {c} candidatas | {row.archivo}')
    pd.DataFrame(control).to_csv(DIR/'control_parseo_2024200501G.csv',index=False)
    pd.DataFrame(descartes,columns=['mes','pagina','archivo','fila_original']).to_csv(DIR/'filas_no_reconocidas_2024200501G.csv',index=False)
    if not todas:raise RuntimeError('Ninguna tabla pudo leerse; revisar formato PDF, no estimar.')
    df=pd.DataFrame(todas).sort_values(['mes','tipo','moneda','denominacion_original','pagina'],kind='stable')
    df.to_csv(RAIZ/'datos_crudos/datos_crudos_2024200501G_aaf.csv',index=False)
    print('Cobertura preliminar:',df.mes.nunique(),'meses;',len(df),'filas; revisar control de parseo.')
    # No se acepta un parser que, en conjunto, omite la mayoría de las líneas candidatas.
    cand=sum(r['candidatos'] for r in control)
    if df.mes.nunique()<24:raise RuntimeError('Menos de 24 meses legibles; revisar control de parseo y no estimar.')
    return df

def main(local=False,demo=False):
    if local:procesar(pd.read_csv(MANIFIESTO));return
    reglas=reglas_portal()
    if demo:
        anterior=pd.read_csv(RAIZ/'datos_crudos/datos_crudos_2024200501G_aaf.csv')
        inventario=pd.read_csv(MANIFIESTO)
        row=inventario[inventario.mes.isin(anterior.mes)].sort_values('mes').iloc[-1]
        b=pedir(row.url,reglas)
        if not b.startswith(b'%PDF'):raise RuntimeError('Respuesta no es PDF')
        import tempfile
        with tempfile.TemporaryDirectory() as temp:
            p=Path(temp)/row.archivo;p.write_bytes(b)
            filas,_,_=leer_pdf(p,row.mes)
            campos=['denominacion_original','moneda','tipo','valor_cuota','rent_mes']
            a=pd.DataFrame(filas)[campos];v=anterior[anterior.mes==row.mes][campos]
            iguales=a.merge(v,on=campos).drop_duplicates()
            print('Filas coincidentes exactas:',len(iguales),'de',len(v),'originales. SHA idéntico:',hashlib.sha256(b).hexdigest()==row.sha256)
            print(iguales.sample(min(10,len(iguales)),random_state=501).to_string(index=False))
        return
    contenido=pedir(LISTADO,reglas);(DIR/'listado_aaf.html').write_bytes(contenido)
    links=enlaces(contenido)
    print('Boletines identificados en catálogo:',len(links))
    if len(links)<24:raise RuntimeError('Catálogo insuficiente; no descargar masivamente ni fabricar meses.')
    registros=[]
    if MANIFIESTO.exists():registros=pd.read_csv(MANIFIESTO).to_dict('records')
    # Primero se prueba un único informe antes de descargar los demás.
    orden=['2025-01']+sorted(m for m in links if m!='2025-01')
    for mes in orden:
        u=links[mes];nombre=f'aaf_boletin_2024200501G_{mes}.pdf';ruta=PDF/nombre
        anterior=next((x for x in registros if x['mes']==mes),None)
        if anterior and ruta.exists() and anterior['url']==u and hashlib.sha256(ruta.read_bytes()).hexdigest()==anterior['sha256']:
            b=ruta.read_bytes()
        else:
            try:
                b=pedir(u,reglas)
            except RuntimeError as e:
                if mes != '2025-01' and str(e).startswith(('HTTP 404;', 'HTTP 410;')):
                    anotar(f'{mes}: enlace no disponible. Se omite.')
                    continue
                raise

            if not b.startswith(b'%PDF'):
                if mes == '2025-01':
                    raise RuntimeError('El piloto no es PDF')
                anotar(f'{mes}: enlace sin PDF. Se omite.')
                continue
            ruta.write_bytes(b)
            registros=[x for x in registros if x['mes']!=mes]
            registros.append({'mes':mes,'archivo':nombre,'url':u,'http':200,'fecha_consulta':datetime.now(timezone(timedelta(hours=-5))).isoformat(),'sha256':hashlib.sha256(b).hexdigest()})
            pd.DataFrame(registros).sort_values('mes').to_csv(MANIFIESTO,index=False)
        if mes=='2025-01':
            f,d,c=leer_pdf(ruta,mes);print(f'PILOTO: {len(f)} filas / {c} candidatas.')
            if len(f)<25 or (c and len(f)/c<.90):raise RuntimeError('El piloto no confirma extracción suficiente. No se descargan los demás boletines.')
            print(pd.DataFrame(f)[['fondo','moneda','tipo','valor_cuota','rent_mes']].head(10).to_string(index=False))
    procesar(pd.DataFrame(registros))

if __name__=='__main__':
    p=argparse.ArgumentParser();g=p.add_mutually_exclusive_group();g.add_argument('--local',action='store_true');g.add_argument('--demo',action='store_true');a=p.parse_args()
    try:main(a.local,a.demo)
    except Exception as e:
        anotar(f'ERROR: {type(e).__name__}: {e}');raise


In [ ]:
correr('02_scraping_web.py', requiere=('01_extraccion_api.py',))

## Paso 5. Limpieza e integración

In [ ]:
%%writefile codigo/03_limpieza_datos.py
# Autora: Amy Pamela Guadalupe Cordova
# Código de matrícula: 2024200501G
# Tema 79: Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
# Fecha de extracción: 2026-11-19
"""Integra AAF (fondo/serie-mes) y BCRP (mes). No modifica crudos ni imputa.
Y: rent_mes. X1: vol12 rezagada. X2: var_tc. X3: retorno del mes anterior.
La rentabilidad rezagada solo usa el mes calendario inmediatamente anterior.
"""
from pathlib import Path
from datetime import datetime,timezone,timedelta
import pandas as pd
import numpy as np
import unicodedata

RAIZ=Path(__file__).resolve().parents[1]; CR=RAIZ/'datos_crudos'; PR=RAIZ/'datos_procesados'; S=RAIZ/'salidas'; PR.mkdir(exist_ok=True);S.mkdir(exist_ok=True)
FECHA_INICIO='2019-01'; FECHA_CORTE='2025-12'; MINIMO=1000
VARIABLES=['rent_mes','vol12','var_tc','rent_lag1']

def anotar(s):
    linea=f'{datetime.now(timezone(timedelta(hours=-5))):%Y-%m-%d %H:%M:%S} | 03_limpieza | {s}'
    print(linea)
    with (RAIZ/'log_ejecucion.txt').open('a',encoding='utf-8') as f:f.write(linea+'\n')

def normalizar(s):
    return ' '.join(''.join(c for c in unicodedata.normalize('NFD',str(s)) if unicodedata.category(c)!='Mn').upper().split())

def main():
    a=pd.read_csv(CR/'datos_crudos_2024200501G_aaf.csv',dtype={'mes':str}).sort_values(['mes','denominacion_original','pagina'])
    b=pd.read_csv(CR/'datos_crudos_2024200501G_bcrp.csv',dtype={'valor_original':str})
    b['fecha']=pd.to_datetime(b.fecha);b['mes']=b.fecha.dt.to_period('M').astype(str)
    b['valor']=pd.to_numeric(b.valor_original,errors='coerce')
    # TC de fin de mes; interbancaria promedio de días observados, en cada moneda.
    m=b.groupby(['mes','variable']).valor.mean().unstack().sort_index().reindex(pd.period_range('2018-12',FECHA_CORTE,freq='M').astype(str))
    m.index.name='mes'
    m['var_tc']=np.log(m.tc_fin.where(m.tc_fin>0)).diff()
    for moneda in ['pen','usd']:
        m['rf_'+moneda]=(1+m['tasa_'+moneda]/100).where(m['tasa_'+moneda]>-100)**(1/12)-1
    m.reset_index().to_csv(S/'tabla_contexto_bcrp.csv',index=False,float_format='%.10g')
    for col in ['valor_cuota','rent_mes']:a[col]=pd.to_numeric(a[col],errors='coerce')
    a.loc[a.valor_cuota.le(0),'valor_cuota']=np.nan
    a['id_fondo_serie']=a.denominacion_original.map(normalizar)+' | '+a.moneda
    filtro=a.mes.between(FECHA_INICIO,FECHA_CORTE)&a.moneda.isin(['PEN','USD'])&a.rent_mes.ge(-1)&a.rent_mes.notna()
    a.loc[~filtro].to_csv(S/'filas_excluidas.csv',index=False)
    a=a.loc[filtro].copy()
    llave=['id_fondo_serie','mes']; valores=['tipo','valor_cuota','rent_mes']
    # Duplicados idénticos, no se suman. Duplicados conflictivos: se registran y excluyen.
    a=a.drop_duplicates(llave+valores)
    conflicto=a.duplicated(llave,keep=False)
    a.loc[conflicto].to_csv(S/'duplicados_conflictivos.csv',index=False)
    a=a.loc[~conflicto].copy()
    a=a.merge(m.reset_index(),on='mes',how='left',validate='many_to_one')
    a['rf_mensual']=np.where(a.moneda.eq('PEN'),a.rf_pen,a.rf_usd)
    a['exceso']=a.rent_mes-a.rf_mensual
    a['vol12']=np.nan
    a['rent_lag1']=np.nan
    # Ventana de 12 meses calendario previos, >=6 retornos observados; no usa el retorno actual.
    for _,g in a.groupby('id_fondo_serie',sort=False):
        s=g.set_index('mes').rent_mes.copy(); s.index=pd.PeriodIndex(s.index,freq='M')
        s=s.reindex(pd.period_range(s.index.min(),s.index.max(),freq='M'))
        v=s.shift(1).rolling(12,min_periods=6).std(ddof=1)
        a.loc[g.index,'vol12']=v.reindex(pd.PeriodIndex(g.mes,freq='M')).to_numpy()
        a.loc[g.index,'rent_lag1']=s.shift(1).reindex(pd.PeriodIndex(g.mes,freq='M')).to_numpy()
    # Atípicos: marca según tipo y moneda; no recorta pérdidas auténticas.
    grupo=a.groupby(['tipo','moneda']).rent_mes
    q1=grupo.transform(lambda s:s.quantile(.25));q3=grupo.transform(lambda s:s.quantile(.75));iqr=q3-q1
    a['atipico']=((iqr>0)&((a.rent_mes<q1-3*iqr)|(a.rent_mes>q3+3*iqr))).astype(int)
    a['imputado']=0
    a['muestra_modelo']=a[VARIABLES].notna().all(axis=1).astype(int)
    a=a.sort_values(llave).reset_index(drop=True)
    muestra=a[a.muestra_modelo.eq(1)]
    cobertura=pd.DataFrame([{'variable':v,'observaciones_validas':int(a[v].notna().sum()),'meses_distintos':int(a.loc[a[v].notna(),'mes'].nunique()),'observaciones_modelo':len(muestra)} for v in VARIABLES])
    cobertura.to_csv(S/'tabla_cobertura_variables.csv',index=False)
    a.groupby('mes').agg(filas=('id_fondo_serie','size'),unidades=('id_fondo_serie','nunique')).reset_index().to_csv(S/'tabla_cobertura_mensual.csv',index=False)
    print(cobertura.to_string(index=False))
    anotar(f'{len(a)} filas | {a.id_fondo_serie.nunique()} fondos/series identificados | {a.mes.nunique()} meses | {len(muestra)} filas completas')
    if (cobertura.observaciones_validas<=MINIMO).any() or len(muestra)<=MINIMO:
        raise RuntimeError('No se supera el mínimo de 1000 filas válidas por variable y completas en el modelo.')
    if muestra.mes.nunique()<24 or muestra.id_fondo_serie.nunique()<10:
        raise RuntimeError('Panel insuficiente para inferencia agrupada: menos de 24 meses o 10 unidades.')
    if a.exceso.notna().sum()<=MINIMO:raise RuntimeError('Cobertura insuficiente de tasas de referencia por moneda.')
    a.to_csv(PR/'datos_procesados_2024200501G.csv',index=False,float_format='%.12g')
    anotar('Base procesada guardada; datos crudos intactos.')

if __name__=='__main__':main()


In [ ]:
correr('03_limpieza_datos.py', requiere=('01_extraccion_api.py', '02_scraping_web.py'))

## Paso 6. Análisis

In [ ]:
%%writefile codigo/04_analisis.py
# Autora: Amy Pamela Guadalupe Cordova
# Código de matrícula: 2024200501G
# Tema 79: Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
# Fecha de extracción: 2026-11-19
"""Desempeño ajustado al riesgo por tipo/moneda y un modelo de tres X.
Inferencia agrupada por fondo/serie y mes; asociación, no causalidad.
"""
from pathlib import Path
from datetime import datetime,timezone,timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

RAIZ=Path(__file__).resolve().parents[1]; S=RAIZ/'salidas';S.mkdir(exist_ok=True)
FORMULA='rent_mes ~ vol12 + var_tc + rent_lag1 + C(tipo) + C(moneda)'

def guardar(d,n):d.to_csv(S/(n+'.csv'),index=False,float_format='%.10g')
def figura(fig,n):
    fig.tight_layout();fig.savefig(S/(n+'.png'),dpi=160,bbox_inches='tight');plt.close(fig)

def estimar(d,nombre):
    ajuste=smf.ols(FORMULA,data=d).fit()
    if np.linalg.matrix_rank(ajuste.model.exog)<ajuste.model.exog.shape[1]:raise RuntimeError('Modelo sin rango completo; revisar variables o categorías.')
    grupos=np.column_stack([pd.factorize(d.id_fondo_serie)[0],pd.factorize(d.mes)[0]])
    robusto=ajuste.get_robustcov_results(cov_type='cluster',groups=grupos,use_correction=True,df_correction=True,use_t=True)
    conf=robusto.conf_int()
    t=pd.DataFrame({'modelo':nombre,'variable':ajuste.model.exog_names,'coeficiente':robusto.params,'error_estandar':robusto.bse,'p_valor':robusto.pvalues,'ic95_inf':conf[:,0],'ic95_sup':conf[:,1],'n':int(ajuste.nobs),'r2':ajuste.rsquared,'fondos_series':d.id_fondo_serie.nunique(),'meses':d.mes.nunique()})
    return ajuste,t

def main():
    d=pd.read_csv(RAIZ/'datos_procesados/datos_procesados_2024200501G.csv',dtype={'mes':str})
    df=d[d.muestra_modelo.eq(1)].copy()
    if len(df)<=1000:raise RuntimeError('Muestra de modelo insuficiente.')
    # Métricas por fondo/serie, tipo y moneda; solo retornos y tasa observados.
    filas=[]
    for (id_,tipo,mon),g in d.dropna(subset=['exceso']).groupby(['id_fondo_serie','tipo','moneda']):
        if len(g)<12:continue
        r=g.rent_mes.to_numpy();e=g.exceso.to_numpy();des=e.std(ddof=1)
        geo=-1. if (r==-1).any() else np.expm1(np.log1p(r).mean()*12)
        filas.append({'id_fondo_serie':id_,'tipo':tipo,'moneda':mon,'n_meses':len(g),'rent_anualizada':geo,'vol_anualizada':r.std(ddof=1)*np.sqrt(12),'sharpe_anualizado':e.mean()/des*np.sqrt(12) if des>0 else np.nan,'meses_con_perdida':(r<0).mean()})
    f=pd.DataFrame(filas)
    if f.empty:raise RuntimeError('Ninguna unidad tiene 12 meses válidos para desempeño.')
    guardar(f,'tabla1_metricas_por_fondo_serie')
    resumen=f.groupby(['tipo','moneda']).agg(unidades=('id_fondo_serie','nunique'),rent_mediana=('rent_anualizada','median'),vol_mediana=('vol_anualizada','median'),sharpe_mediana=('sharpe_anualizado','median'),meses_perdida_mediana=('meses_con_perdida','median')).reset_index()
    guardar(resumen,'tabla2_riesgo_rentabilidad_por_tipo')
    # MCO; corrección de inferencia, no modificación de coeficientes. Dos dimensiones de agrupación.
    inicial,principal=estimar(df,'principal_cluster_fondo_mes')
    guardar(principal,'tabla3_modelo_principal')
    clasico=pd.DataFrame({'variable':inicial.model.exog_names,'coeficiente':inicial.params.to_numpy(),'error_estandar_mco':inicial.bse.to_numpy(),'p_valor_mco':inicial.pvalues.to_numpy()})
    guardar(clasico,'tabla4_modelo_inicial_mco')
    alt=df[df.atipico.eq(0)].copy();rob=principal.copy()
    if len(alt)>1000 and alt.mes.nunique()>=24 and alt.id_fondo_serie.nunique()>=10:
        try:_,t=estimar(alt,'sensibilidad_sin_atipicos');rob=pd.concat([rob,t],ignore_index=True)
        except RuntimeError as e:print('Robustez no estimable:',e)
    guardar(rob,'tabla5_robustez')
    vif=pd.DataFrame({'variable':inicial.model.exog_names,'VIF':[variance_inflation_factor(inicial.model.exog,i) for i in range(inicial.model.exog.shape[1])]})
    guardar(vif,'tabla6_vif')
    # Correlación residual rezagada descriptiva dentro de cada serie; no apilar para aplicar DW/ADF.
    resid=df[['id_fondo_serie','mes']].copy();resid['residuo']=inicial.resid.to_numpy();resid=resid.sort_values(['id_fondo_serie','mes'])
    resid['mes_num']=pd.PeriodIndex(resid.mes,freq='M').asi8
    g=resid.groupby('id_fondo_serie');resid['rezago']=g.residuo.shift();resid.loc[resid.mes_num-g.mes_num.shift()!=1,'rezago']=np.nan
    guardar(pd.DataFrame([{'n':len(df),'fondos_series':df.id_fondo_serie.nunique(),'meses':df.mes.nunique(),'corr_residuos_lag1':resid[['residuo','rezago']].corr().iloc[0,1],'advertencia':'Correlación descriptiva; errores agrupados, no prueba causal.'}]),'tabla7_diagnosticos')
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.2})
    fig,axs=plt.subplots(1,2,figsize=(11,4),sharey=True)
    for ax,mon in zip(axs,['PEN','USD']):
        for tipo,gg in f[f.moneda.eq(mon)].groupby('tipo'):ax.scatter(gg.vol_anualizada*100,gg.rent_anualizada*100,label=tipo,s=18,alpha=.6)
        ax.set(title=mon,xlabel='Volatilidad anualizada (%)',ylabel='Rentabilidad anualizada (%)')
        if ax.has_data():ax.legend(fontsize=7,loc='best')
    figura(fig,'figura1_riesgo_rentabilidad')
    fig,axs=plt.subplots(1,2,figsize=(11,4),sharey=True)
    for ax,mon in zip(axs,['PEN','USD']):
        a=resumen[resumen.moneda.eq(mon)].sort_values('sharpe_mediana');ax.barh(a.tipo,a.sharpe_mediana,color='#315c80');ax.set(title=mon,xlabel='Mediana del Sharpe anualizado')
    figura(fig,'figura2_sharpe_por_tipo')
    mensual=d.groupby(['mes','tipo','moneda']).rent_mes.mean().reset_index();guardar(mensual,'tabla8_rentabilidades_mensuales')
    fig,axs=plt.subplots(2,1,figsize=(11,7),sharex=True)
    for ax,mon in zip(axs,['PEN','USD']):
        for tipo,g in mensual[mensual.moneda.eq(mon)].groupby('tipo'):
            ax.plot(pd.to_datetime(g.mes+'-01'),100*g.rent_mes,label=tipo,lw=1)
        ax.set(title=mon,ylabel='Rentabilidad mensual (%)');ax.legend(fontsize=7,ncol=3)
    figura(fig,'figura3_rentabilidad_mensual')
    fig,ax=plt.subplots(figsize=(7,4));x=principal[principal.variable.isin(['vol12','var_tc','rent_lag1'])].dropna(subset=['ic95_inf','ic95_sup'])
    if not x.empty:ax.errorbar(x.coeficiente,x.variable,xerr=[x.coeficiente-x.ic95_inf,x.ic95_sup-x.coeficiente],fmt='o',color='#315c80',capsize=4)
    ax.axvline(0,color='gray',lw=1);ax.set(xlabel='Coeficiente e intervalo de confianza al 95 %',title='Modelo principal: tres factores');figura(fig,'figura4_coeficientes')
    print(resumen.to_string(index=False));print(principal.to_string(index=False))
    with (RAIZ/'log_ejecucion.txt').open('a',encoding='utf-8') as ar:ar.write(f'{datetime.now(timezone(timedelta(hours=-5))):%Y-%m-%d %H:%M:%S} | 04_analisis | {len(df)} filas; tablas y figuras guardadas\n')

if __name__=='__main__':main()


In [ ]:
correr('04_analisis.py', requiere=('03_limpieza_datos.py',))

## Paso 7. Visualizar y documentar resultados

In [ ]:
if ESTADO.get('04_analisis.py')!=0: raise RuntimeError('Primero termina el análisis')
from IPython.display import display, Image
import csv, itertools
for p in sorted(Path('salidas').glob('tabla[1-8]_*.csv')):
    print(p.name)
    with p.open(encoding='utf-8',newline='') as archivo:
        for fila in itertools.islice(csv.reader(archivo),12):print(' | '.join(fila))
for p in sorted(Path('salidas').glob('figura*.png')):
    display(Image(filename=str(p),width=850))

## Paso 8. Verificación reproducible (quinto script)

In [ ]:
%%writefile codigo/05_verificacion_fuente.py
# Autora: Amy Pamela Guadalupe Cordova
# Código de matrícula: 2024200501G
# Tema 79: Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
# Fecha de extracción: 2026-11-19
"""Por defecto verifica la entrega y la reproduce en una copia temporal sin internet.
--en-vivo: además llama a los modos demo de 01 y 02, sin sobrescribir originales.
"""
from pathlib import Path
import argparse, hashlib, shutil, subprocess, sys, tempfile
import numpy as np
import pandas as pd

RAIZ=Path(__file__).resolve().parents[1]

def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def equivalencia(a,b):
    if sha(a)==sha(b):return 1.0
    x=pd.read_csv(a,keep_default_na=False,dtype=str);y=pd.read_csv(b,keep_default_na=False,dtype=str)
    if x.shape!=y.shape or list(x.columns)!=list(y.columns):return 0.0
    if x.empty:return 1.0
    # Las claves y fechas deben coincidir exactamente; para números, tolerancia de redondeo.
    for k in ['id_fondo_serie','mes','fecha','codigo','denominacion_original','moneda','tipo','variable','archivo']:
        if k in x and not x[k].equals(y[k]):return 0.0
    eq=x.eq(y)
    for col in x:
        n=pd.to_numeric(x[col],errors='coerce');m=pd.to_numeric(y[col],errors='coerce')
        eq[col]=eq[col] | (n.notna()&m.notna()&np.isclose(n,m,rtol=1e-9,atol=1e-11))
    return float(eq.to_numpy().mean())

def main(vivo=False):
    registro=[]
    rutas=[]
    for linea in (RAIZ/'hash_sha256.txt').read_text(encoding='utf-8').splitlines():
        esperado,rel=linea.split('  ',1);p=RAIZ/rel
        if not p.is_file() or sha(p)!=esperado:raise RuntimeError('Hash no coincide con archivo entregado: '+rel)
        rutas.append(rel)
    if not rutas or 'datos_procesados/datos_procesados_2024200501G.csv' not in rutas:raise RuntimeError('Manifiesto incompleto')
    registro.append(f'Integridad original: {len(rutas)} archivos verificados antes de regenerar.')
    with tempfile.TemporaryDirectory(prefix='repro_finanzas_') as tmp:
        t=Path(tmp)
        for carpeta in ['codigo','datos_crudos']:shutil.copytree(RAIZ/carpeta,t/carpeta)
        for carpeta in ['datos_procesados','salidas']: (t/carpeta).mkdir()
        for rel in ['datos_crudos/datos_crudos_2024200501G_bcrp.csv','datos_crudos/datos_crudos_2024200501G_aaf.csv','datos_crudos/scraping/control_parseo_2024200501G.csv','datos_crudos/scraping/filas_no_reconocidas_2024200501G.csv']:
            (t/rel).unlink(missing_ok=True)
        orden=[('01_extraccion_api.py','--local'),('02_scraping_web.py','--local'),('03_limpieza_datos.py',),('04_analisis.py',)]
        for args in orden:subprocess.run([sys.executable,str(t/'codigo'/args[0]),*args[1:]],cwd=t,check=True)
        # Reparseo de originales y regeneración de tablas: no se reescribe la entrega.
        for rel in rutas:
            if not rel.endswith('.csv'):continue
            if not (t/rel).is_file():raise RuntimeError('Archivo CSV no reproducido: '+rel)
            q=equivalencia(RAIZ/rel,t/rel)
            registro.append(f'{rel}: equivalencia {q:.2%}')
            if q<.95:raise RuntimeError(f'{rel}: equivalencia inferior al 95 %')
        originales={p.name for p in (RAIZ/'salidas').glob('figura*.png')};nuevos={p.name for p in (t/'salidas').glob('figura*.png')}
        if not originales or originales!=nuevos:raise RuntimeError('No se regeneraron todas las figuras.')
        registro.append(f'{len(originales)} figuras regeneradas; se contrastaron las tablas (PNG puede variar por plataforma).')
    d=pd.read_csv(RAIZ/'datos_procesados/datos_procesados_2024200501G.csv')
    d.sample(10,random_state=501).to_csv(RAIZ/'salidas/muestra_cotejo_2024200501G.csv',index=False)
    registro.append('APROBADO: reproducción computacional local. Esto no equivale a un cotejo nuevo contra la fuente viva.')
    (RAIZ/'salidas/verificacion_local.md').write_text('\n\n'.join(registro)+'\n',encoding='utf-8');print('\n'.join(registro))
    if vivo:
        for s in ['01_extraccion_api.py','02_scraping_web.py']:
            subprocess.run([sys.executable,str(RAIZ/'codigo'/s),'--demo'],cwd=RAIZ,check=True)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--en-vivo',action='store_true');p.add_argument('--sin-internet',action='store_true');a=p.parse_args()
    if a.en_vivo and a.sin_internet:p.error('Elige una modalidad')
    main(a.en_vivo)


### Documentación y hash de los archivos originales, antes de reproducir

In [ ]:
import hashlib, re
if ESTADO.get('04_analisis.py')!=0: raise RuntimeError('No se generan evidencias de una ejecución incompleta')
log=Path('log_ejecucion.txt').read_text(encoding='utf-8')
fechas_api=re.findall(r'^(\d{4}-\d{2}-\d{2}).*\| 01_api \| GUARDADO',log,re.M)
if not fechas_api: raise RuntimeError('No consta extracción real API en el log')
import csv
def leer_csv(ruta):
    with open(ruta,encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
manifest=leer_csv('datos_crudos/scraping/descargas_boletines_2024200501G.csv')
f_aaf=sorted(r['fecha_consulta'][:10] for r in manifest)
fecha_extraccion=max(fechas_api[-1],f_aaf[-1])
for p in Path('codigo').glob('*.py'):
    lineas=p.read_text(encoding='utf-8').splitlines(keepends=True)
    lineas[3]=f'# Fecha de extracción: {fecha_extraccion}\n'
    p.write_text(''.join(lineas),encoding='utf-8')
# Huellas de respuestas originales, bases y resultados generados (no incluye el propio informe 05).
archivos=[p for nombre in ['datos_crudos','datos_procesados','salidas'] for p in Path(nombre).rglob('*') if p.is_file() and p.name not in ['verificacion_local.md','muestra_cotejo_2024200501G.csv']]
Path('hash_sha256.txt').write_text(''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.as_posix()}\n' for p in sorted(archivos)),encoding='utf-8')
d=leer_csv('datos_procesados/datos_procesados_2024200501G.csv')
cob=leer_csv('salidas/tabla_cobertura_variables.csv')
meses_observados=sorted({r['mes'] for r in d})
n_unidades=len({r['id_fondo_serie'] for r in d})
n_modelo=sum(int(r['muestra_modelo']) for r in d)
sha=hashlib.sha256(Path('datos_procesados/datos_procesados_2024200501G.csv').read_bytes()).hexdigest()
mes_inicio,mes_fin=meses_observados[0],meses_observados[-1]
meses_es='enero febrero marzo abril mayo junio julio agosto septiembre octubre noviembre diciembre'.split()
def fecha_apa(s):
    x=datetime.fromisoformat(s);return f'{x.day} de {meses_es[x.month-1]} de {x.year}'
referencias=f"""Banco Central de Reserva del Perú. (s. f.). *BCRPData: Series estadísticas de tipo de cambio e interbancarias* [Conjunto de datos]. Recuperado el {fecha_apa(fechas_api[-1])}, de https://estadisticas.bcrp.gob.pe/estadisticas/series/

Asociación de Administradoras de Fondos del Perú. (s. f.). *Estadísticas mensuales de fondos mutuos* [Informes estadísticos]. Recuperado el {fecha_apa(f_aaf[-1])}, de https://fondosmutuos.pe/estadisticas/ (consultas registradas del {fecha_apa(f_aaf[0])} al {fecha_apa(f_aaf[-1])})."""
Path('README.md').write_text(f"""# Fondos mutuos en el Perú: rentabilidad y riesgo por tipo de fondo
Amy Pamela Guadalupe Cordova · 2024200501G · UNCP · Finanzas I (055D) · Unidad II · Tema 79

Objetivo: comparar desempeño ajustado por riesgo. Periodo objetivo 2019-01 a 2025-12; observado {mes_inicio} a {mes_fin}, {len(meses_observados)} meses; {len(d)} filas de fondo/serie-mes; {n_unidades} denominaciones identificadas; {n_modelo} filas completas de estimación. No son fondos únicos verificados por código registral. Modelo Y = rent_mes, X1=vol12, X2=var_tc, X3=rent_lag1, controles de tipo y moneda; errores agrupados por serie y mes. El número de meses independientes es menor que el número de filas del panel. No se dispone de patrimonio y partícipes individuales de forma homogénea ni de rentabilidad diaria. Ver metodología y límites en el cuaderno. No se declara autorización de sustitución sin constancia.

## Fuentes y endpoints
API BCRP: `https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{{codigo}}/json/{{inicio}}/{{fin}}/esp`; PN01216PM, PD04692MD, PD04693MD. Fechas reales en log. AAF: `https://fondosmutuos.pe/estadisticas/`, PDF exactos y hashes en `datos_crudos/scraping/descargas_boletines_2024200501G.csv`. Llave: mes, armonización desde días efectivos de tasas. Boletines PDF originales intactos. Rutas relativas a cada script. Python {sys.version.split()[0]}, versiones exactas en requirements.txt.

## Reproducción por el docente (sin consultar fuentes)
```
git clone https://github.com/AmyGuadalupe/Finanzas-Segundo-Trabajo.git
cd Finanzas-Segundo-Trabajo
python -m pip install -r requirements.txt
python codigo/05_verificacion_fuente.py --sin-internet
```
05 coteja hash de lo entregado antes de ejecutar, copia originales a un directorio temporal, ejecuta 01 y 02 localmente para reparsear JSON/PDF, 03 y 04 para regenerar la base y tablas. 03 y 04 son ejecutables independientemente. No necesita token ni Drive. Para una nueva extracción, usar otra copia del repositorio y ejecutar 01, 02, 03 y 04 en orden; no modificar originales entregados. Demostración en vivo: `python codigo/01_extraccion_api.py --demo` y `python codigo/02_scraping_web.py --demo`, solo si el acceso está permitido. No confundir esta demostración con la reproducción local.

## Integridad
SHA-256 de la base: `{sha}`. Manifiesto `hash_sha256.txt`. Cobertura por variable: `salidas/tabla_cobertura_variables.csv`. Tablas y figuras en `salidas/`; cada archivo empírico es generado por código. Integridad y equivalencia se verifican en el paso siguiente; no se anticipa el veredicto.

## Fuentes en formato APA
{referencias}

Repositorio: https://github.com/AmyGuadalupe/Finanzas-Segundo-Trabajo
""",encoding='utf-8')
Path('diccionario_variables.md').write_text('# Diccionario de variables\nFrecuencia final mensual; clave `id_fondo_serie` + `mes`. AAF: https://fondosmutuos.pe/estadisticas/ (URL de cada PDF en manifiesto). BCRP: https://estadisticas.bcrp.gob.pe/estadisticas/series/api/ (código exacto abajo).\n\n| Variable | Definición | Unidad / origen | Rol |\n|---|---|---|---|\n| mes | Mes del boletín | AAAA-MM, AAF | Tiempo |\n| tipo | Clasificación de la tabla de origen | Categoría, AAF | Control y agrupación |\n| fondo, administradora | Nombre reportado; administradora cuando se reconoce | Texto, AAF | Identificación |\n| denominacion_original | Denominación original, incluye administradora cuando figura | Texto, AAF | Evidencia |\n| id_fondo_serie | Denominación normalizada + moneda | Texto, calculada | Llave |\n| moneda | Moneda de la cuota | PEN/USD, AAF | Control |\n| estado | No informado por el boletín | Texto | Evidencia, no filtro |\n| valor_cuota | Valor publicado de cuota | Moneda del fondo, AAF | Descriptivo |\n| rent_mes | Rentabilidad mensual publicada | Decimal, AAF | Y |\n| rent_lag1 | Rentabilidad del mes calendario previo, faltante si no existe | Decimal, calculada | X3 |\n| tc_fin | Tipo cambio interbancario promedio fin de periodo | PEN/USD, BCRP PN01216PM | Insumo |\n| tasa_pen / tasa_usd | Media mensual de observaciones diarias interbancarias | % anual, BCRP PD04692MD / PD04693MD | Insumo |\n| var_tc | Diferencia logarítmica del TC fin de mes | Decimal, calculada | X2 |\n| rf_pen / rf_usd | (1+tasa/100)^(1/12)-1 | Decimal mensual, calculada | Referencia |\n| rf_mensual | Referencia de la moneda del fondo | Decimal, calculada | Insumo |\n| exceso | rent_mes - rf_mensual | Decimal, calculada | Sharpe |\n| vol12 | Desv. estándar muestral de los 12 meses anteriores; mínimo 6 | Decimal mensual, calculada | X1 |\n| atipico | Fuera de Q1–3RIQ a Q3+3RIQ por tipo y moneda | 0/1, calculada | Robustez |\n| imputado | No se imputan valores | 0, calculada | Control |\n| muestra_modelo | Y y X1–X3 disponibles | 0/1, calculada | Control |\n| archivo / pagina / fila_original | Ubicación y texto leído del PDF | Texto/número, AAF | Trazabilidad |\n',encoding='utf-8')
Path('documentos/bitacora_ia.md').write_text('La preparación técnica del notebook se realizó con asistencia de ChatGPT, utilizando como referencia el código de la Unidad I, la versión 0.3 y la propuesta de informes AAF de Claude. La estudiante deberá registrar sus ejecuciones, revisión, correcciones y los usos reales posteriores. No se atribuyen pruebas o resultados no efectuados.\n',encoding='utf-8')
for fila in cob:print(fila)
print('SHA-256 base:',sha)

In [ ]:
correr('05_verificacion_fuente.py', requiere=('03_limpieza_datos.py','04_analisis.py'), argumentos=('--sin-internet',))

## Paso 9. Guardar el notebook y descargar ZIP de Carpeta 3
Si los scripts y la verificación están completos, se descarga el ZIP. Guarda también la copia del cuaderno desde Archivo → Descargar `.ipynb`. No se crea un ZIP de la investigación completa hasta que existan productos y desarrollo.

In [ ]:
import shutil, zipfile
from google.colab import files
if ESTADO.get('05_verificacion_fuente.py')!=0: raise RuntimeError('Verificación pendiente; no se ofrece entrega completa')
# Copia del cuaderno actual, si Colab permite recuperar su contenido.
try:
    from google.colab import _message
    respuesta=_message.blocking_request('get_ipynb',timeout_sec=30)
    contenido=respuesta['ipynb']
    if isinstance(contenido,str):contenido=json.loads(contenido)
    # Sincronizar las cuatro primeras líneas de cada %%writefile con el script generado.
    for c in contenido['cells']:
        s=''.join(c.get('source','')); primero=s.split('\n',1)[0]
        if primero.startswith('%%writefile codigo/'):
            nombre=primero.split(' ',1)[1]
            if Path(nombre).exists():c['source']=(primero+'\n'+Path(nombre).read_text(encoding='utf-8')).splitlines(keepends=True)
    # Los resultados se conservan en salidas. No copiar posibles salidas sensibles del notebook.
    for c in contenido['cells']:
        if c.get('cell_type')=='code':c['outputs']=[];c['execution_count']=None
    Path('codigo/00_pipeline_colab.ipynb').write_text(json.dumps(contenido,ensure_ascii=False,indent=1),encoding='utf-8')
except Exception as e:
    print('Colab no devolvió la copia del notebook. Descárgalo desde Archivo y añádelo a codigo/ antes de la entrega.')
zip_path=Path('/content')/(RAIZ.name+'.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RAIZ.rglob('*')):
        if p.is_file() and '__pycache__' not in p.parts and p.name!='.env':z.write(p,RAIZ.name+'/'+p.relative_to(RAIZ).as_posix())
with zipfile.ZipFile(zip_path) as z:
    if z.testzip():raise RuntimeError('ZIP dañado')
print(zip_path.name, round(zip_path.stat().st_size/1024**2,2),'MB')
files.download(str(zip_path))

## Paso 10. GitHub (opcional en la ejecución automática; obligatorio para la entrega)
Publicación consciente: activa la celda únicamente cuando quieras publicar. Token `GITHUB-TOKEN` en Secretos. El repositorio debe contener el contenido de Carpeta 3 en su raíz. Tres commits en fechas distintas, reales. La descarga del ZIP ya ocurrió. No se borran repositorios remotos automáticamente.

In [ ]:
PUBLICAR = False  # cambiar a True solo al publicar
MENSAJE = 'datos y codigo: extraccion y analisis'  # cambia en cada avance real
if PUBLICAR:
    from google.colab import userdata
    import tempfile, shutil
    url='https://github.com/AmyGuadalupe/Finanzas-Segundo-Trabajo.git'
    repo=Path('/content/Finanzas-Segundo-Trabajo')
    if not repo.exists():subprocess.run(['git','clone',url,str(repo)],check=True)
    if subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()!=url:raise RuntimeError('URL remota inesperada')
    subprocess.run(['git','-C',str(repo),'config','user.name','Amy Guadalupe'],check=True)
    subprocess.run(['git','-C',str(repo),'config','user.email','e_2024200501g@uncp.edu.pe'],check=True)
    # El clon se usa solo para publicar. No se importan archivos del repositorio al pipeline.
    rastreados=subprocess.check_output(['git','-C',str(repo),'ls-files'],text=True).splitlines()
    sobrantes=[x for x in rastreados if not (RAIZ/x).is_file() and x not in ('README.md','.gitignore','LICENSE')]
    if sobrantes:raise RuntimeError('El repositorio contiene archivos ajenos a esta ejecución. Revísalos antes de publicar: '+str(sobrantes[:8]))
    for p in RAIZ.rglob('*'):
        if not p.is_file() or '__pycache__' in p.parts or p.name=='.env':continue
        if p.stat().st_size>95*1024**2:raise RuntimeError('Archivo demasiado grande para GitHub: '+p.name)
        destino=repo/p.relative_to(RAIZ);destino.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,destino)
    subprocess.run(['git','-C',str(repo),'add','.'],check=True)
    subprocess.run(['git','-C',str(repo),'status','--short'],check=True)
    if input('Revisa los archivos. Escribe PUBLICAR para commit y push: ').strip()=='PUBLICAR':
        token=userdata.get('GITHUB-TOKEN')
        if not token:raise RuntimeError('Falta GITHUB-TOKEN en Secretos')
        if subprocess.run(['git','-C',str(repo),'diff','--cached','--quiet']).returncode==1:
            subprocess.run(['git','-C',str(repo),'commit','-m',MENSAJE],check=True)
        # GIT_ASKPASS evita que el token aparezca en URL o config. Se elimina el auxiliar temporal.
        with tempfile.TemporaryDirectory() as temp:
            ask=Path(temp)/'askpass.sh';ask.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "AmyGuadalupe";; *) echo "$GITHUB_TOKEN";; esac\n');ask.chmod(0o700)
            env=os.environ.copy();env.update({'GITHUB_TOKEN':token,'GIT_ASKPASS':str(ask),'GIT_TERMINAL_PROMPT':'0'})
            resultado=subprocess.run(['git','-C',str(repo),'push'],env=env,capture_output=True,text=True)
            print((resultado.stdout+resultado.stderr).replace(token,'***'))
            if resultado.returncode:raise RuntimeError('Push falló; revisa mensaje. No uses force push.')
        del token,env
        subprocess.run(['git','-C',str(repo),'log','-5','--format=%ad %h %s','--date=short'],check=True)

## Demostración en vivo (solo para la exposición)
No forma parte del flujo automático. Si el profesor lo solicita, ejecuta manualmente los comandos de abajo, una vez entregados los crudos y únicamente si la fuente permite el acceso. Los programas no sobrescriben el archivo original.

```python
!python codigo/01_extraccion_api.py --demo
!python codigo/02_scraping_web.py --demo
```

La entrega completa debe conservar la organización de Productos, Desarrollo y Base de datos y código. Los productos académicos en Typst y el dashboard se elaboran después con las salidas auténticas. No se incluyen documentos vacíos como si fueran entregables terminados.